# How to Train a Custom Keypoint Detection Model with PyTorch

## 1. Imports

In [ ]:
import os, json, cv2, numpy as np, matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.transforms import functional as F

import albumentations as A

from IPython.display import display, clear_output

In [ ]:
from detection import transforms, utils, engine
from detection.utils import collate_fn
from detection.engine import train_one_epoch, evaluate

## 2. Augmentations

In [ ]:
def train_transform():
    return A.Compose([
        A.Sequential([
            A.RandomRotate90(p=1), # Random rotation of an image by 90 degrees zero or more times
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, brightness_by_max=True, always_apply=False, p=1), # Random change of brightness & contrast
        ], p=1)
    ],
    keypoint_params=A.KeypointParams(format='xy'),
    bbox_params=A.BboxParams(format='pascal_voc', label_fields=['bboxes_labels'])
    )

## 3. Dataset class

In [ ]:
class ClassDataset(Dataset):
    def __init__(self, root, transform=None, demo=False):
        self.root = root
        self.transform = transform
        self.demo = demo
        self.imgs_files = sorted(os.listdir(os.path.join(root, "images")))
        self.annotations_files = sorted(os.listdir(os.path.join(root, "labels")))

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "images", self.imgs_files[idx])
        annotations_path = os.path.join(self.root, "labels", self.annotations_files[idx])

        img_original = cv2.imread(img_path)
        img_original = cv2.cvtColor(img_original, cv2.COLOR_BGR2RGB)

        with open(annotations_path) as f:
            data = json.load(f)
            bboxes_original = data['bboxes']
            keypoints_original = data['keypoints']
            bboxes_labels_original = ['Vertebra' for _ in bboxes_original]

        if self.transform:
            keypoints_original_flattened = [el[0:2] for kp in keypoints_original for el in kp]
            transformed = self.transform(image=img_original, bboxes=bboxes_original, bboxes_labels=bboxes_labels_original, keypoints=keypoints_original_flattened)
            img = transformed['image']
            bboxes = transformed['bboxes']
            keypoints_transformed_unflattened = np.reshape(np.array(transformed['keypoints']), (-1,2,2)).tolist()

            keypoints = []
            for o_idx, obj in enumerate(keypoints_transformed_unflattened):
                obj_keypoints = []
                for k_idx, kp in enumerate(obj):
                    obj_keypoints.append(kp + [keypoints_original[o_idx][k_idx][2]])
                keypoints.append(obj_keypoints)

        else:
            img, bboxes, keypoints = img_original, bboxes_original, keypoints_original

        bboxes = torch.as_tensor(bboxes, dtype=torch.float32)
        target = {}
        target["boxes"] = bboxes
        target["labels"] = torch.as_tensor([1 for _ in bboxes], dtype=torch.int64)
        target["image_id"] = idx
        target["area"] = (bboxes[:, 3] - bboxes[:, 1]) * (bboxes[:, 2] - bboxes[:, 0])
        target["iscrowd"] = torch.zeros(len(bboxes), dtype=torch.int64)
        target["keypoints"] = torch.as_tensor(keypoints, dtype=torch.float32)
        img = F.to_tensor(img)

        bboxes_original = torch.as_tensor(bboxes_original, dtype=torch.float32)
        target_original = {}
        target_original["boxes"] = bboxes_original
        target_original["labels"] = torch.as_tensor([1 for _ in bboxes_original], dtype=torch.int64)
        target_original["image_id"] = idx
        target_original["area"] = (bboxes_original[:, 3] - bboxes_original[:, 1]) * (bboxes_original[:, 2] - bboxes_original[:, 0])
        target_original["iscrowd"] = torch.zeros(len(bboxes_original), dtype=torch.int64)
        target_original["keypoints"] = torch.as_tensor(keypoints_original, dtype=torch.float32)
        img_original = F.to_tensor(img_original)

        if self.demo:
            return img, target, img_original, target_original
        else:
            return img, target

    def __len__(self):
        return len(self.imgs_files)

## 4. Visualizing a random item from dataset

In [ ]:
KEYPOINTS_FOLDER_TRAIN = '../dataset/fold1/treino'
dataset = ClassDataset(KEYPOINTS_FOLDER_TRAIN, transform=train_transform(), demo=True)
data_loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

iterator = iter(data_loader)
batch = next(iterator)

print("Original targets:\n", batch[3], "\n\n")
print("Transformed targets:\n", batch[1])

In [ ]:
keypoints_classes_ids2names = {0: 'C2', 1: 'C4'}

def visualize(image, bboxes, keypoints, image_original=None, bboxes_original=None, keypoints_original=None):
    fontsize = 18

    for bbox in bboxes:
        start_point = (bbox[0], bbox[1])
        end_point = (bbox[2], bbox[3])
        image = cv2.rectangle(image.copy(), start_point, end_point, (0,255,0), 2)

    for kps in keypoints:
        for idx, kp in enumerate(kps):
            image = cv2.circle(image.copy(), tuple(kp), 3, (255,0,0), 3)
            image = cv2.putText(image.copy(), " " + keypoints_classes_ids2names[idx], tuple(kp), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1, cv2.LINE_AA)

    if image_original is None and keypoints_original is None:
        plt.figure(figsize=(20,20))
        plt.imshow(image)

    else:
        for bbox in bboxes_original:
            start_point = (bbox[0], bbox[1])
            end_point = (bbox[2], bbox[3])
            image_original = cv2.rectangle(image_original.copy(), start_point, end_point, (0,255,0), 2)

        for kps in keypoints_original:
            for idx, kp in enumerate(kps):
                image_original = cv2.circle(image_original, tuple(kp), 3, (255,0,0), 3)
                image_original = cv2.putText(image_original, " " + keypoints_classes_ids2names[idx], tuple(kp), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,0,0), 1, cv2.LINE_AA)

        f, ax = plt.subplots(1, 2, figsize=(20, 10))

        ax[0].imshow(image_original)
        ax[0].set_title('Original image', fontsize=fontsize)

        ax[1].imshow(image)
        ax[1].set_title('Transformed image', fontsize=fontsize)

In [ ]:
image = (batch[0][0].permute(1,2,0).numpy() * 255).astype(np.uint8)
bboxes = batch[1][0]['boxes'].detach().cpu().numpy().astype(np.int32).tolist()

keypoints = []
for kps in batch[1][0]['keypoints'].detach().cpu().numpy().astype(np.int32).tolist():
    keypoints.append([kp[:2] for kp in kps])

image_original = (batch[2][0].permute(1,2,0).numpy() * 255).astype(np.uint8)
bboxes_original = batch[3][0]['boxes'].detach().cpu().numpy().astype(np.int32).tolist()

keypoints_original = []
for kps in batch[3][0]['keypoints'].detach().cpu().numpy().astype(np.int32).tolist():
    keypoints_original.append([kp[:2] for kp in kps])

visualize(image, bboxes, keypoints, image_original, bboxes_original, keypoints_original)

## 5. Training

In [ ]:
def get_model(num_keypoints, weights_path=None):

    anchor_generator = AnchorGenerator(sizes=(32, 64, 128, 256, 512), aspect_ratios=(0.25, 0.5, 0.75, 1.0, 2.0, 3.0, 4.0))
    model = torchvision.models.detection.keypointrcnn_resnet50_fpn(pretrained=False,
                                                                   pretrained_backbone=True,
                                                                   num_keypoints=num_keypoints,
                                                                   num_classes=2,
                                                                   rpn_anchor_generator=anchor_generator)

    if weights_path:
        state_dict = torch.load(weights_path)
        model.load_state_dict(state_dict)

    return model

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Configuration
FOLD_NUMBER = 1
USE_AUGMENTATION = True  # Set to False to disable augmentation

# Folder names based on configuration
aug_suffix = "with_aug" if USE_AUGMENTATION else "no_aug"
experiment_name = f"fold{FOLD_NUMBER}_{aug_suffix}"

KEYPOINTS_FOLDER_TRAIN = f'../dataset/fold{FOLD_NUMBER}/treino'
KEYPOINTS_FOLDER_TEST = f'../dataset/fold{FOLD_NUMBER}/teste'
# Apply augmentation based on configuration
train_aug = train_transform() if USE_AUGMENTATION else None

dataset_train = ClassDataset(KEYPOINTS_FOLDER_TRAIN, transform=train_aug, demo=False)
dataset_test = ClassDataset(KEYPOINTS_FOLDER_TEST, transform=None, demo=False)

data_loader_train = DataLoader(dataset_train, batch_size=16, shuffle=True, collate_fn=collate_fn)
data_loader_test = DataLoader(dataset_test, batch_size=4, shuffle=False, collate_fn=collate_fn)

model = get_model(num_keypoints=2)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.8)

# Number of epochs
num_epochs = 50

# Lists to store metrics
train_losses = []
val_losses = []
val_bbox_ap = []
val_bbox_ar = []
val_keypoint_ap = []
val_keypoint_ar = []

# Create figure for real-time plots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for epoch in range(num_epochs):
    # Training
    metric_logger = train_one_epoch(model, optimizer, data_loader_train, device, epoch, print_freq=1000)
    train_losses.append(metric_logger.meters['loss'].global_avg)

    # Evaluation
    coco_evaluator = evaluate(model, data_loader_test, device)

    # Calculate val_loss
    model.train()
    val_loss_total = 0.0
    with torch.no_grad():
        for images, targets in data_loader_test:
            images = list(img.to(device) for img in images)
            targets = [{k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            val_loss_total += sum(loss for loss in loss_dict.values()).item()
    val_losses.append(val_loss_total / len(data_loader_test))
    model.eval()

    lr_scheduler.step()

    # Extract metrics from COCO evaluator
    val_bbox_ap.append(coco_evaluator.coco_eval['bbox'].stats[0])
    val_bbox_ar.append(coco_evaluator.coco_eval['bbox'].stats[8])
    val_keypoint_ap.append(coco_evaluator.coco_eval['keypoints'].stats[0])
    val_keypoint_ar.append(coco_evaluator.coco_eval['keypoints'].stats[5])

    # Calculate F1 scores
    val_bbox_f1 = [2*ap*ar/(ap+ar) if (ap+ar) > 0 else 0 for ap, ar in zip(val_bbox_ap, val_bbox_ar)]
    val_keypoint_f1 = [2*ap*ar/(ap+ar) if (ap+ar) > 0 else 0 for ap, ar in zip(val_keypoint_ap, val_keypoint_ar)]

    # Update real-time plots
    epochs_range = range(1, len(train_losses) + 1)
    current_lr = optimizer.param_groups[0]['lr']

    clear_output(wait=True)
    for ax in axes.flat:
        ax.clear()

    # Train Loss + Val Loss
    axes[0, 0].plot(epochs_range, train_losses, 'b-o', label='Train Loss', markersize=3)
    axes[0, 0].plot(epochs_range, val_losses, 'r-o', label='Val Loss', markersize=3)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    # Average Precision
    axes[0, 1].plot(epochs_range, val_bbox_ap, 'g-o', label='BBox AP', markersize=3)
    axes[0, 1].plot(epochs_range, val_keypoint_ap, 'purple', marker='o', label='KP AP', markersize=3)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('AP')
    axes[0, 1].set_title('Average Precision')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    axes[0, 1].set_ylim(0, 1)

    # Average Recall
    axes[1, 0].plot(epochs_range, val_bbox_ar, 'g-s', label='BBox AR', markersize=3)
    axes[1, 0].plot(epochs_range, val_keypoint_ar, 'purple', marker='s', label='KP AR', markersize=3)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('AR')
    axes[1, 0].set_title('Average Recall')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    axes[1, 0].set_ylim(0, 1)

    # F1 Score
    axes[1, 1].plot(epochs_range, val_bbox_f1, 'g-D', label='BBox F1', markersize=3)
    axes[1, 1].plot(epochs_range, val_keypoint_f1, 'purple', marker='D', label='KP F1', markersize=3)
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('F1')
    axes[1, 1].set_title('F1 Score')
    axes[1, 1].legend()
    axes[1, 1].grid(True)
    axes[1, 1].set_ylim(0, 1)

    plt.suptitle(f'{experiment_name} | Epoch {epoch+1}/{num_epochs} | KP F1: {val_keypoint_f1[-1]:.4f} | Best KP F1: {max(val_keypoint_f1):.4f} | LR: {current_lr:.5f}')
    plt.tight_layout()
    display(fig)

# Create output folder
output_folder = f'../models/{experiment_name}'
os.makedirs(output_folder, exist_ok=True)

# Save model weights
model_path = f'{output_folder}/keypointsrcnn_weights_{experiment_name}.pth'
torch.save(model.state_dict(), model_path)

# Save final plot
plot_path = f'{output_folder}/training_plot_{experiment_name}.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')

# Métricas finais
print(f"\n{'='*60}")
print(f"Training Complete! ({experiment_name})")
print(f"{'='*60}")
print(f"Final BBox  - AP: {val_bbox_ap[-1]:.4f}, AR: {val_bbox_ar[-1]:.4f}, F1: {val_bbox_f1[-1]:.4f}")
print(f"Final KP    - AP: {val_keypoint_ap[-1]:.4f}, AR: {val_keypoint_ar[-1]:.4f}, F1: {val_keypoint_f1[-1]:.4f}")
print(f"Best KP F1: {max(val_keypoint_f1):.4f} (Epoch {val_keypoint_f1.index(max(val_keypoint_f1)) + 1})")
print(f"\nSaved to: {output_folder}/")
print(f"  - Model: keypointsrcnn_weights_{experiment_name}.pth")
print(f"  - Plot:  training_plot_{experiment_name}.png")

## 6. Visualizing model predictions

In [ ]:
iterator = iter(data_loader_test)

In [ ]:
images, targets = next(iterator)
images = list(image.to(device) for image in images)

with torch.no_grad():
    model.to(device)
    model.eval()
    output = model(images)

print("Predictions: \n", output)

In [ ]:
image = (images[0].permute(1,2,0).detach().cpu().numpy() * 255).astype(np.uint8)
scores = output[0]['scores'].detach().cpu().numpy()

high_scores_idxs = np.where(scores > 0.5)[0].tolist()
post_nms_idxs = torchvision.ops.nms(output[0]['boxes'][high_scores_idxs], output[0]['scores'][high_scores_idxs], 0.3).cpu().numpy()

keypoints = []
for kps in output[0]['keypoints'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    keypoints.append([list(map(int, kp[:2])) for kp in kps])

bboxes = []
for bbox in output[0]['boxes'][high_scores_idxs][post_nms_idxs].detach().cpu().numpy():
    bboxes.append(list(map(int, bbox.tolist())))

visualize(image, bboxes, keypoints)